# 05 — Modeling and Evaluation

This notebook builds baseline machine learning pipelines for customer churn prediction.

Main goals:
- prepare train and test sets
- build reproducible preprocessing pipelines
- compare multiple classification models
- evaluate model performance with suitable classification metrics

In [83]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from pathlib import Path
import json

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)



## 1. Load the dataset and configuration

The raw dataset is loaded again and the saved preprocessing decisions are reused to keep the modeling stage consistent with the previous notebook.

In [84]:
DATA_PATH = Path("../data/raw/telco.csv")

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

Shape: (7043, 50)


,Customer ID,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Country,State,...,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Satisfaction Score,Customer Status,Churn Label,Churn Score,CLTV,Churn Category,Churn Reason
0,8779-QRDMV,Male,78,No,Yes,No,No,0,United States,California,...,20,0.00,59.65,3,Churned,Yes,91,5433,Competitor,Competitor offered more data
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,United States,California,...,0,390.80,1024.10,3,Churned,Yes,69,5302,Competitor,Competitor made better offer
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,United States,California,...,0,203.94,1910.88,2,Churned,Yes,81,3179,Competitor,Competitor made better offer
3,4598-XLKNJ,Female,78,No,Yes,Yes,Yes,1,United States,California,...,0,494.00,2995.07,2,Churned,Yes,88,5337,Dissatisfaction,Limited range of services
4,4846-WHAFZ,Female,80,No,Yes,Yes,Yes,1,United States,California,...,0,234.21,3102.36,2,Churned,Yes,67,2793,Price,Extra data charges


In [85]:
CONFIG_PATH = Path("../reports/feature_config.json")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    feature_config = json.load(f)

feature_config

{'target_column': 'target_churn',
 'dropped_columns': ['Customer Status',
  'Churn Category',
  'Churn Reason',
  'Churn Score',
  'Customer ID',
  'Country',
  'State',
  'City',
  'Zip Code',
  'Latitude',
  'Longitude',
  'Quarter',
  'CLTV',
  'Total Revenue',
  'Under 30',
  'Senior Citizen',
  'Dependents',
  'Referred a Friend',
  'Churn Label'],
 'numeric_features': ['Age',
  'Number of Dependents',
  'Population',
  'Number of Referrals',
  'Tenure in Months',
  'Avg Monthly Long Distance Charges',
  'Avg Monthly GB Download',
  'Monthly Charge',
  'Total Charges',
  'Total Refunds',
  'Total Extra Data Charges',
  'Total Long Distance Charges',
  'Satisfaction Score'],
 'categorical_features': ['Gender',
  'Married',
  'Offer',
  'Phone Service',
  'Multiple Lines',
  'Internet Service',
  'Internet Type',
  'Online Security',
  'Online Backup',
  'Device Protection Plan',
  'Premium Tech Support',
  'Streaming TV',
  'Streaming Movies',
  'Streaming Music',
  'Unlimited Data

## 2. Recreate the baseline modeling dataset

The target variable and the preprocessing decisions from the previous notebook are applied again here.

The saved configuration helps keep column exclusions and feature groups consistent, while the raw dataset is prepared again to keep this notebook reproducible and self-contained.

In [86]:
df["target_churn"] = df["Churn Label"].map({"No": 0, "Yes": 1})

object_cols = df.select_dtypes(include="object").columns.tolist()
df[object_cols] = df[object_cols].replace("None", np.nan)
df[object_cols] = df[object_cols].replace(r"^\s*$", np.nan, regex=True)

# semantic missing-value fixes from preprocessing
if {"Internet Service", "Internet Type"}.issubset(df.columns):
    no_internet_mask = df["Internet Service"].eq("No") & df["Internet Type"].isna()
    df.loc[no_internet_mask, "Internet Type"] = "No Internet"

if "Internet Type" in df.columns:
    df["Internet Type"] = df["Internet Type"].fillna("Missing")

if "Offer" in df.columns:
    df["Offer"] = df["Offer"].fillna("No Offer")

df[["Churn Label", "target_churn"]].head()

,Churn Label,target_churn
0,Yes,1
1,Yes,1
2,Yes,1
3,Yes,1
4,Yes,1


In [87]:
print(df["Internet Type"].value_counts(dropna=False))
print()
print(df["Offer"].value_counts(dropna=False))

Internet Type
Fiber Optic    3035
DSL            1652
No Internet    1526
Cable           830
Name: count, dtype: int64

Offer
No Offer    3877
Offer B      824
Offer E      805
Offer D      602
Offer A      520
Offer C      415
Name: count, dtype: int64


In [88]:
drop_cols = feature_config["dropped_columns"]

baseline_df = df.drop(columns=drop_cols).copy()

print("Baseline shape:", baseline_df.shape)
baseline_df.head()

Baseline shape: (7043, 32)


,Gender,Age,Married,Number of Dependents,Population,Number of Referrals,Tenure in Months,Offer,Phone Service,Avg Monthly Long Distance Charges,...,Contract,Paperless Billing,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Satisfaction Score,target_churn
0,Male,78,No,0,68701,0,1,No Offer,No,0.00,...,Month-to-Month,Yes,Bank Withdrawal,39.65,39.65,0.00,20,0.00,3,1
1,Female,74,Yes,1,55668,1,8,Offer E,Yes,48.85,...,Month-to-Month,Yes,Credit Card,80.65,633.30,0.00,0,390.80,3,1
2,Male,71,No,3,47534,0,18,Offer D,Yes,11.33,...,Month-to-Month,Yes,Bank Withdrawal,95.45,1752.55,45.61,0,203.94,2,1
3,Female,78,Yes,1,27778,1,25,Offer C,Yes,19.76,...,Month-to-Month,Yes,Bank Withdrawal,98.50,2514.50,13.43,0,494.00,2,1
4,Female,80,Yes,1,26265,1,37,Offer C,Yes,6.33,...,Month-to-Month,Yes,Bank Withdrawal,76.50,2868.15,0.00,0,234.21,2,1


In [90]:
feature_config["dropped_columns"]

['Customer Status',
 'Churn Category',
 'Churn Reason',
 'Churn Score',
 'Customer ID',
 'Country',
 'State',
 'City',
 'Zip Code',
 'Latitude',
 'Longitude',
 'Quarter',
 'CLTV',
 'Total Revenue',
 'Under 30',
 'Senior Citizen',
 'Dependents',
 'Referred a Friend',
 'Churn Label']

## 2.1 Define features and target

The baseline dataset is split into input features (`X`) and target labels (`y`) for modeling.

In [91]:
X = baseline_df.drop(columns=["target_churn"])
y = baseline_df["target_churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 31)
y shape: (7043,)


## 2.2 Identify feature groups

Numeric and categorical feature groups are loaded from the saved preprocessing configuration.

These groups will be used to apply different preprocessing steps inside the modeling pipeline.

In [92]:
numeric_features = feature_config["numeric_features"]
categorical_features = feature_config["categorical_features"]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nNumeric columns:")
print(numeric_features)

print("\nCategorical columns:")
print(categorical_features)

Numeric features: 13
Categorical features: 18

Numeric columns:
['Age', 'Number of Dependents', 'Population', 'Number of Referrals', 'Tenure in Months', 'Avg Monthly Long Distance Charges', 'Avg Monthly GB Download', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Satisfaction Score']

Categorical columns:
['Gender', 'Married', 'Offer', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method']


## 3. Train-test split

The dataset is divided into training and test sets.

The test set is reserved for final evaluation, while model comparison is supported with cross-validation on the training set.

In [93]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5634, 31)
X_test : (1409, 31)
y_train: (5634,)
y_test : (1409,)


## 4. Cross-validation setup

To make the model comparison more reliable, cross-validation is applied on the training set.

This allows the baseline models to be compared more robustly before the final test-set evaluation.

In [94]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

### Why cross-validation was used instead of a separate validation split

A separate validation set could have been created, but this project uses cross-validation on the training data instead.

Because the dataset is moderate in size, this approach allows more efficient use of the available training observations while still providing a more stable comparison across models.

The test set is kept untouched and reserved only for final evaluation.

## 5. Build preprocessing pipelines

Separate preprocessing steps are defined for numeric and categorical features.

These are combined using a `ColumnTransformer` to create a clean and reproducible workflow.

In [95]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

In [96]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

categorical_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore'))])

In [97]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Age', 'Number of Dependents', 'Population',
                                  'Number of Referrals', 'Tenure in Months',
                                  'Avg Monthly Long Distance Charges',
                                  'Avg Monthly GB Download', 'Monthly Charge',
                                  'Total Charges', 'Total Refunds',
                                  'Total Extra Data Charges',
                                  'Total Long Distanc...
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['Gender', 'Married', 'Offer', 'Phone Service',
                                  'Multiple Lines', 'Internet Service',
                                  'Internet Type', 'Online Security',
                                  'Online Backup', 'Device Protection Plan',
                                  'Premium Tech Support', 'Streaming TV',
                                  'Streaming Movies', 'Streaming Music',
                                  'Unlimited Data', 'Contract',
                                  'Paperless Billing', 'Payment Method'])])

## Numeric Features
* **Median Imputation:** Chosen for robustness against outliers compared to mean imputation.
* **StandardScaler:** Applied to normalize feature magnitudes, ensuring a stable baseline for linear models.

## Categorical Features
* **Most Frequent (Mode) Imputation:** A simple baseline strategy for missing data.
* **One-Hot Encoding:** Used to convert categories to numeric vectors without introducing artificial ordinality.

## Decision Matrix

| Stage | Method | Rationale |
| :--- | :--- | :--- |
| **Num Impute** | Median | Robustness |
| **Num Scale** | StandardScaler | Feature parity |
| **Cat Impute** | Most Frequent | Simplicity |
| **Cat Encode** | One-Hot | No artificial order |


## 6. Define baseline models

The following baseline models are compared:

- **Logistic Regression:** Serves as a linear baseline, highly interpretable.
- **Decision Tree:** Captures non-linear patterns with high transparency.
- **Random Forest:** An ensemble method (Bagging) that reduces variance and improves stability.
- **XGBoost:** A gradient boosting method (Boosting) that iteratively optimizes performance, often yielding the highest accuracy for tabular data.

These models represent different modeling families, ensuring that our baseline comparison is comprehensive and robust.

In [98]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric="logloss")
}

models

{'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
 'Decision Tree': DecisionTreeClassifier(random_state=42),
 'Random Forest': RandomForestClassifier(random_state=42),
 'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric='logloss',
               feature_types=None, feature_weights=None, gamma=None,
               grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=None, max_bin=None,
               max_cat_threshold=None, max_cat_to_onehot=None,
               max_delta_step=None, max_depth=None, max_leaves=None,
               min_child_weight=None, missing=nan, monotone_constraints=None,
               multi_strategy=None, n_estimators=None, n_jobs=None,
               num_parallel_tree=None, ...)}

## 7. Create an evaluation helper function

A reusable function `evaluate_model` is implemented to streamline the evaluation process. This ensures that every model follows the exact same preprocessing and validation protocol.

**Key Features:**
- **Automated Pipeline:** Combines preprocessing and the estimator to prevent data leakage.
- **Cross-Validation:** Uses F1-score as the primary metric during CV to account for potential class imbalance.
- **Comprehensive Metrics:** Calculates Accuracy, Precision, Recall, F1-score, and ROC-AUC for a 360-degree performance view.
- **Robustness:** Includes checks for probability prediction availability to ensure compatibility across different model types.

In [99]:
def evaluate_model(model_name, model, preprocessor, X_train, X_test, y_train, y_test, cv):
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])
    
    cv_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="f1"
    )
    
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    
    if hasattr(pipeline, "predict_proba"):
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        y_proba = None
        roc_auc = np.nan
    
    results = {
        "model": model_name,
        "cv_f1_mean": cv_scores.mean(),
        "cv_f1_std": cv_scores.std(),
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_roc_auc": roc_auc
    }
    
    return pipeline, y_pred, y_proba, results

## 8. Train and evaluate all baseline models

Each model is trained through the same preprocessing pipeline.

Cross-validated training performance and final test-set metrics are collected for comparison.

In [100]:
all_results = []
trained_pipelines = {}

for model_name, model in models.items():
    pipeline, y_pred, y_proba, results = evaluate_model(
        model_name=model_name,
        model=model,
        preprocessor=preprocessor,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        cv=cv
    )
    
    trained_pipelines[model_name] = {
        "pipeline": pipeline,
        "y_pred": y_pred,
        "y_proba": y_proba
    }
    
    all_results.append(results)

## 9. Compare model performance

The baseline models are compared using cross-validated F1-score and final test-set metrics.

This helps identify which model provides the strongest starting point for further tuning and interpretation.

In [101]:
results_df = pd.DataFrame(all_results).sort_values(by="cv_f1_mean", ascending=False)
results_df.round(4)

,model,cv_f1_mean,cv_f1_std,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
2,Random Forest,0.9272,0.0071,0.9567,0.9700,0.8636,0.9137,0.9832
3,XGBoost,0.9228,0.0032,0.9624,0.9547,0.9011,0.9271,0.9901
0,Logistic Regression,0.9222,0.0081,0.9567,0.9484,0.8850,0.9156,0.9912
1,Decision Tree,0.8928,0.0051,0.9468,0.9118,0.8850,0.8982,0.9271


In [102]:
best_cv_model_name = results_df.iloc[0]["model"]
best_cv_model_name

'Random Forest'

In [103]:
# Selected as the preferred baseline model after comparing cross-validation
# performance, held-out test metrics, and interpretability.
preferred_model_name = "Logistic Regression"
preferred_model_name

'Logistic Regression'

## Baseline model selection note

Among the baseline models, tree-based methods achieved slightly stronger performance in cross-validation and on the test set.

However, Logistic Regression was kept as the preferred baseline model because its performance remained competitive while offering a much clearer interpretation of feature effects. At this stage, it serves as a strong reference point before moving to tuning and conservative feature review.

In [ ]:
selected_y_pred = trained_pipelines[preferred_model_name]["y_pred"]

print(f"Preferred model: {preferred_model_name}\n")
print(classification_report(y_test, selected_y_pred))

Preferred model: Logistic Regression

              precision    recall  f1-score   support

           0       0.96      0.98      0.97      1035
           1       0.95      0.89      0.92       374

    accuracy                           0.96      1409
   macro avg       0.95      0.93      0.94      1409
weighted avg       0.96      0.96      0.96      1409



## Interpretation of the preferred baseline model

At the baseline stage, Logistic Regression gave a clean and fairly balanced result.

The confusion matrix shows that the model handled both classes reasonably well. It correctly identified most non-churned customers and most churned customers, while keeping false positives and false negatives relatively low.

For the churn class, the model reached:

- **Precision = 0.95**
- **Recall = 0.90**
- **F1-score = 0.93**

These results are strong for a baseline model. I still do not want to treat them as final, especially because some retained features, such as `Satisfaction Score`, may be contributing a lot of predictive power.

Even so, Logistic Regression works well here as a baseline reference because it stays competitive and is much easier to interpret than the more complex models.

In [ ]:
selected_cm = confusion_matrix(y_test, selected_y_pred)
selected_cm

array([[1017,   18],
       [  43,  331]], dtype=int64)

## Conclusion

This notebook moved from preprocessing into the first round of model evaluation.

The baseline results show that tree-based models perform slightly better overall, while Logistic Regression remains a strong reference model because it stays competitive and is easier to interpret.

At the same time, these results should not be treated as final. Some retained variables, especially `Satisfaction Score`, may be adding substantial predictive power. For that reason, the next stage will focus on tuning, stricter comparison, and a more conservative version of the feature set.